In [3]:
import torch 
import torch.nn as nn 
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import re
#sample data
texts = [
    "this movie was absolutely amazing",
    "i loved every moment of this film",
    "great acting and wonderful story",
    "best movie i have ever seen",
    "highly recommend this masterpiece",
    "this was terrible and boring",
    "worst movie i have ever watched",
    "bad acting and poor story",
    "complete waste of time",
    "i hated everything about this film",
    "not good at all very disappointing",
    "awful experience never watching again",

]
labels = [1,1,1,1,1 ,0,0,0,0,0,0,0]
def tokenize(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]','',text)
    return text.split()
def build_vocab(texts,max_vocab= 1000):
    counter = Counter()
    for text in texts:
        counter.update(tokenize(text))
    vocab = {'<PAD>': 0,'<UNK>': 1}
    for word,count in counter.most_common(max_vocab - 2):
        vocab[word] = len(vocab)
    return vocab
vocab = build_vocab(texts)
print(f"vocab size : {len(vocab)}")

def encode(text,vocab,max_len=20):
    tokens = tokenize(text)
    ids = [vocab.get(t,1) for t in tokens]
    ids = ids[:max_len]
    ids = ids + [0] * (max_len - len(ids))
    return ids
print(encode("this movie was amazing", vocab))
class TextDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=20):
        self.data   = [encode(t, vocab, max_len) for t in texts]
        self.labels = labels

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return (torch.tensor(self.data[idx], dtype=torch.long),
                torch.tensor(self.labels[idx], dtype=torch.float))
dataset = TextDataset(texts, labels, vocab)
loader  = DataLoader(dataset, batch_size=4, shuffle=True)

    
class SentimentLSTM(nn.Module):
    def __init__(self,vocab_size,embed_dim,hidden_size,num_layers):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings = vocab_size,
            embedding_dim = embed_dim,
            padding_idx = 0
        )
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_size,
            num_layers = num_layers,
            batch_first = True,
            dropout = 0.3
        )
        self.fc = nn.Linear(hidden_size, 1)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        embedded = self.embedding(x)
        out, _ = self.lstm(embedded)
        out = out[:, -1, :]
        out = self.fc(out)
        out = self.sigmoid(out)
        return out.squeeze(1)
    
device = "cuda" if torch.cuda.is_available() else "cpu"
model     = SentimentLSTM(
    vocab_size   = len(vocab),
    embed_dim    = 64,
    hidden_size  = 128,
    num_layers   = 2
).to(device)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(),lr = 0.001)
print(f"paramters : {sum(p.numel() for p in model.parameters()):,}")
epochs = 20
print("\n" + "="*45)
print(f"{'Epoch':<8}{'Loss':>10}{'Accuracy':>12}")
print("="*45)

for epoch in range(epochs):
    model.train()
    total_loss = correct = total = 0
    for texts_batch,labels_batch in loader:
        texts_batch = texts_batch.to(device)
        labels_batch = labels_batch.to(device)
        outputs = model(texts_batch)
        loss = criterion(outputs,labels_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        predicted = (outputs > 0.5).float()
        correct += (predicted == labels_batch).sum().item()
        total += labels_batch.size(0)
    avg_loss = total_loss / len(loader)
    accuracy = correct / total * 100
    if (epoch + 1) % 5 ==0:
        print(f"{epoch+1:<8}{avg_loss:>10.4f}{accuracy:>11.2f}%")
print("="*45)
model.eval()
test_reviews = [
    "this film was absolutely wonderful",
    "terrible movie complete waste of time",
    "amazing story loved every second",
    "this is good but not perfect ",
]
print("\nPredictions: ")
print("-"*45)
with torch.no_grad():
    for review in test_reviews:
        encoded = torch.tensor(
            [encode(review, vocab)], dtype=torch.long
        ).to(device)
        prob = model(encoded).item()
        label = "POSITIVE 😊" if prob > 0.5 else "NEGATIVE 😡"
        print(f"{label} ({prob:.2f}) | {review}")


vocab size : 48
[2, 4, 6, 14, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
paramters : 234,625

Epoch         Loss    Accuracy
5           0.6850      58.33%
10          0.6802      58.33%
15          0.6584      58.33%
20          0.1494     100.00%

Predictions: 
---------------------------------------------
POSITIVE 😊 (0.76) | this film was absolutely wonderful
NEGATIVE 😡 (0.00) | terrible movie complete waste of time
POSITIVE 😊 (0.76) | amazing story loved every second
NEGATIVE 😡 (0.00) | this is good but not perfect 
